## Imports

In [1]:
import os
import re

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)
local_datasets = '/Users/rapido/local-datasets/customer/appography/appography_v0/raw/'
print(local_datasets)

/Users/rapido/analytics/customer/Appography/Appography_v0
/Users/rapido/local-datasets/customer/appography/appography_v0/raw/


### customer_installed_apps

Scaling started from 20240219

##customer_installed_apps

customer_installed_apps_query = f"""

    SELECT
        *
    FROM
        (
        SELECT
            json_extract_scalar(data, '$.userId') as user,
            id, 
            yyyymmdd,
            json_extract_scalar(data, '$.city') as city,
            json_extract(data, '$.appographyData') as apps_list,
            row_number() over(partition by id order by updated_epoch desc) updated_seq,
            *
        FROM 
            raw_internal.kafka_info_customer_installed_apps_v1
        WHERE 
            yyyymmdd >= DATE_FORMAT((DATE_TRUNC('month', CURRENT_DATE) - INTERVAL '2' MONTH), '%Y%m%d')
            AND yyyymmdd <= DATE_FORMAT((CURRENT_DATE - INTERVAL '1' DAY), '%Y%m%d')
        )
    WHERE
        updated_seq = 1
"""

df_customer_installed_apps = pd.read_sql(customer_installed_apps_query, connection)
df_customer_installed_apps.to_csv(local_datasets + 'raw_customer_installed_apps.csv', index=False)

In [5]:
df_customer_installed_apps_raw = pd.read_csv(local_datasets + 'raw_customer_installed_apps.csv')
df_customer_installed_apps_raw.head(3)

,user,id,yyyymmdd,city,apps_list,updated_seq,id.1,hhmmss,updated_hhmmss,epoch,updated_epoch,metadata,quarter_hour,updated_quarter_hour,updated_yyyymmdd,event_epoch,data,yyyymmdd.1,hh
0,65d66f036e7643db9f5fe3b0,65d66f036e7643db9f5fe3b0,20240222,Delhi,"[""Adobe Scan"","" CRED"","" Facebook"","" Messenger""...",1,65d66f036e7643db9f5fe3b0,31609,31609,1708551969571,1708551969571,"{""epochFieldsMapping"":{""epoch"":""$.createdOn"",""...",315,315,20240222,1708551969571,"{""hh"":""03"",""event_quarter_hour"":""0315"",""metada...",20240222,3
1,63b185bc825607d4143320a1,63b185bc825607d4143320a1,20240222,NaN,"[""Clock"","" Facebook"","" Messenger"","" Google Map...",1,63b185bc825607d4143320a1,30242,30242,1708551162356,1708551162356,"{""epochFieldsMapping"":{""epoch"":""$.createdOn"",""...",300,300,20240222,1708551162356,"{""hh"":""03"",""event_quarter_hour"":""0300"",""metada...",20240222,3
2,64fcbb70573694ef75bbe72e,64fcbb70573694ef75bbe72e,20240222,NaN,"[""Zomato"","" Facebook"","" Messenger"","" Google Ma...",1,64fcbb70573694ef75bbe72e,35504,35504,1708554304845,1708554304845,"{""epochFieldsMapping"":{""epoch"":""$.createdOn"",""...",345,345,20240222,1708554304845,"{""hh"":""03"",""event_quarter_hour"":""0345"",""metada...",20240222,3


In [6]:
df_customer_installed_apps = df_customer_installed_apps_raw[['user', 'apps_list', 'city', 'metadata']].copy()

# Define a function to extract values between double quotations and trim extra spaces
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r'"(.*?)"', text)]

df_customer_installed_apps['apps_list'] = df_customer_installed_apps['apps_list'].apply(lambda x: extract_and_trim(x))

In [7]:
df_customer_installed_apps

,user,apps_list,city,metadata
0,65d66f036e7643db9f5fe3b0,"[Adobe Scan, CRED, Facebook, Messenger, Goibib...",Delhi,"{""epochFieldsMapping"":{""epoch"":""$.createdOn"",""..."
1,63b185bc825607d4143320a1,"[Clock, Facebook, Messenger, Google Maps, Goog...",NaN,"{""epochFieldsMapping"":{""epoch"":""$.createdOn"",""..."
2,64fcbb70573694ef75bbe72e,"[Zomato, Facebook, Messenger, Google Maps, Goo...",NaN,"{""epochFieldsMapping"":{""epoch"":""$.createdOn"",""..."
3,5f7acff50fadf216fa5cf5ea,"[Clock, Axis Mobile, Wynk Music, CRED, Google ...",NaN,"{""epochFieldsMapping"":{""epoch"":""$.createdOn"",""..."
4,628840c18f9b43effa5f9551,"[Clock, BookMyShow, Facebook, Google Sheets, G...",Bangalore,"{""epochFieldsMapping"":{""epoch"":""$.createdOn"",""..."
...,...,...,...,...
1071457,63be37b0c2c0bf139a497509,"[Clock, Google News, Google Maps, Google Photo...",NaN,"{""epochFieldsMapping"":{""epoch"":""$.createdOn"",""..."
1071458,627b572a87b0a6c007c59a8b,"[Adobe Lightroom, Clock, Zomato, BigBasket, Bo...",Hyderabad,"{""epochFieldsMapping"":{""epoch"":""$.createdOn"",""..."
1071459,646b2555db888ce1197cf322,"[Adobe Scan, Clock, Zomato, Facebook, Google A...",Jaipur,"{""epochFieldsMapping"":{""epoch"":""$.createdOn"",""..."
1071460,65107a2e1213a92b7b9fe4ed,"[Wynk Music, CRED, Facebook, Messenger, Google...",Kolkata,"{""epochFieldsMapping"":{""epoch"":""$.createdOn"",""..."


### category_list

https://docs.google.com/spreadsheets/d/1O6IsJ2dzzhXKuoHFoOA0AD4s4SOerMC1U-KrpeFnbwQ/edit#gid=1742529009

category_list = pd.read_clipboard()
category_list.to_csv(local_datasets + 'category_list_v1.csv', index=False)

In [8]:
category_list = pd.read_csv(local_datasets + 'category_list_v1.csv')
category_list = category_list[['app_name', 'categories_l2', 'categories_l1']]

In [9]:
category_list.head(2)

,app_name,categories_l2,categories_l1
0,IGNOU e-Content,Educational,Educational
1,Aha,Entertainment,OTT


In [10]:
category_list.categories_l1.unique()

array(['Educational', 'OTT', 'Delivery_Grocery', 'Tools', 'Office',
       'Dating', 'News', 'Devotional', 'Travel_Metro', 'Gaming',
       'Travel_Bookings', 'Finance_Transactions', 'Ecommerce',
       'Streaming_Music', 'Finance_Investment', 'Health',
       'Fantasy_Sports', 'Delivery_Food', 'Home_Security', 'Finance_News',
       'Travel_Ridehailing', 'Social', 'Telecom', 'Entertainment',
       'News_finance', 'Wearable', 'Driver_App'], dtype=object)

In [11]:
category_list.categories_l2.unique()

array(['Educational', 'Entertainment', 'Delivery', 'Tools', 'Office',
       'News', 'Devotional', 'Travel', 'Gaming', 'Finance', 'Ecommerce',
       'Health', 'Home_Security', 'Social', 'Telecom', 'Wearable',
       'Driver_App'], dtype=object)

### Active customer (RR-customers)

In [12]:
## Parameter 
start_date = '20240213'
end_date = '20240222'

##active_customer_query

active_customer_query = f"""

WITH active_customers AS (

    SELECT 
        channel_host,
        customer_id,
        city_name
    FROM 
        (
        SELECT 
            channel_host,
            customer_id,
            city_name,
            row_number() over(partition by customer_id order by updated_epoch DESC) updated_seq
        FROM 
            orders.order_logs_snapshot
        WHERE
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND channel_host IN ('android', 'ios')
        )
    WHERE
        updated_seq = 1
    ), 
    
    iallocator_customer_segments AS (
    
    SELECT
        customer_id user_id,
        taxi_lifetime_rides ltr,
        taxi_ltr_segment ltr_segment,
        taxi_recency_segment recency_segment,
        taxi_frequency_segment frequency_segment,
        taxi_retention_segments retention_segment,
        taxi_income_segment income_segment,
        taxi_service_affinity service_affinity,
        customer_service_segments customer_service_segment,
        ps_tag_auto,
        ps_tag_link
    FROM 
        datasets.iallocator_customer_segments
    WHERE 
        run_date = '2024-02-22'
        AND customer_id IN (SELECT customer_id FROM active_customers)
    )
    
    SELECT
        a.*,
        ltr,
        CASE 
        WHEN ltr = 0 THEN 'LTR 0'
        WHEN ltr > 0 THEN ltr_segment
        END AS ltr_segment,
        recency_segment,
        frequency_segment,
        retention_segment,
        income_segment,
        service_affinity,
        customer_service_segment,
        ps_tag_auto,
        ps_tag_link
    FROM 
        active_customers a
    LEFT JOIN 
        iallocator_customer_segments b
        ON a.customer_id = b.user_id
"""

df_active_customer = pd.read_sql(active_customer_query, connection)
df_active_customer.to_csv(local_datasets + 'df_active_customer.csv', index=False)

In [13]:
df_active_customer = pd.read_csv(local_datasets + 'df_active_customer.csv')
df_active_customer.head(3)

,channel_host,customer_id,city_name,ltr,ltr_segment,recency_segment,frequency_segment,retention_segment,income_segment,service_affinity,customer_service_segment,ps_tag_auto,ps_tag_link
0,android,65572a2517233209a025edaa,Chennai,25.0,PHH,RECENT,PERFORMANT,PLATINUM,HIGH_INCOME,ONLY_AUTO,BOTH,NPS,NPS
1,android,5e416ded47cd7059b5950bcb,Delhi,196.0,PHH,RECENT,PERFORMANT,PRIME,MEDIUM_INCOME,ONLY_LINK,BOTH,UNKNOWN,PS
2,android,5c4ee43a4a267149c77dc2e1,Bangalore,110.0,PHH,DORMANT,PERFORMANT,DORMANT,MEDIUM_INCOME,ONLY_LINK,BOTH,UNKNOWN,UNKNOWN


## Data workaround

In [14]:
print(df_active_customer.shape)
df_active_customer.head(3)

(7211344, 13)


,channel_host,customer_id,city_name,ltr,ltr_segment,recency_segment,frequency_segment,retention_segment,income_segment,service_affinity,customer_service_segment,ps_tag_auto,ps_tag_link
0,android,65572a2517233209a025edaa,Chennai,25.0,PHH,RECENT,PERFORMANT,PLATINUM,HIGH_INCOME,ONLY_AUTO,BOTH,NPS,NPS
1,android,5e416ded47cd7059b5950bcb,Delhi,196.0,PHH,RECENT,PERFORMANT,PRIME,MEDIUM_INCOME,ONLY_LINK,BOTH,UNKNOWN,PS
2,android,5c4ee43a4a267149c77dc2e1,Bangalore,110.0,PHH,DORMANT,PERFORMANT,DORMANT,MEDIUM_INCOME,ONLY_LINK,BOTH,UNKNOWN,UNKNOWN


In [15]:
print(df_customer_installed_apps.shape)
df_customer_installed_apps.head(3)

(1071462, 4)


,user,apps_list,city,metadata
0,65d66f036e7643db9f5fe3b0,"[Adobe Scan, CRED, Facebook, Messenger, Goibib...",Delhi,"{""epochFieldsMapping"":{""epoch"":""$.createdOn"",""..."
1,63b185bc825607d4143320a1,"[Clock, Facebook, Messenger, Google Maps, Goog...",NaN,"{""epochFieldsMapping"":{""epoch"":""$.createdOn"",""..."
2,64fcbb70573694ef75bbe72e,"[Zomato, Facebook, Messenger, Google Maps, Goo...",NaN,"{""epochFieldsMapping"":{""epoch"":""$.createdOn"",""..."


In [16]:
print(category_list.shape)
category_list.head(3)

(399, 3)


,app_name,categories_l2,categories_l1
0,IGNOU e-Content,Educational,Educational
1,Aha,Entertainment,OTT
2,Grofers delivery app,Delivery,Delivery_Grocery


### What is the distribution of mobile os among our customer base?

In [17]:
df_temp = df_active_customer.groupby(['channel_host']).agg({'customer_id':'nunique'}).reset_index()
df_temp['% distribuion'] = (df_temp['customer_id']*100/df_temp['customer_id'].sum()).round(2)
df_temp

,channel_host,customer_id,% distribuion
0,android,6063041,84.08
1,ios,1148303,15.92


### Which city should I choose?

In [18]:
df_customer_coverage = pd.merge(df_active_customer[df_active_customer['channel_host'].isin(['android'])],
                                df_customer_installed_apps,
                                how='left',
                                left_on=['customer_id'], 
                                right_on=['user']
                                )

df_temp = df_customer_coverage\
            .groupby(['city_name'])\
            .agg(active_customers = pd.NamedAgg('customer_id','nunique'),
                 app_info_customers = pd.NamedAgg('user','nunique')
                )\
            .sort_values(['active_customers'],ascending=False)\
            .reset_index()
df_temp['coverage %'] = (df_temp['app_info_customers']*100/df_temp['active_customers']).round(2)
df_temp.head(10)

,city_name,active_customers,app_info_customers,coverage %
0,Hyderabad,1463077,147456,10.08
1,Bangalore,950563,98655,10.38
2,Delhi,825640,102589,12.43
3,Chennai,661041,68864,10.42
4,Jaipur,205449,24874,12.11
5,Kolkata,200985,23574,11.73
6,Pune,124580,13437,10.79
7,Ahmedabad,119218,15182,12.73
8,Indore,116684,14014,12.01
9,Lucknow,112497,13416,11.93


### Proceeding with our plans focusing on Hyderabad city, with a few customer metrics in mind.

In [19]:
## Merge Active customers & customers apps details

df_customer_hyd_all = df_customer_coverage[(df_customer_coverage['city_name'].isin(['Hyderabad'])) 
                                       &
                                       (df_customer_coverage['user'].notna())]
df_customer_hyd = df_customer_hyd_all[['customer_id', 'apps_list', 
                                       'ltr', 'ltr_segment'
#                                        'recency_segment', 'income_segment',
#                                        'frequency_segment', 'retention_segment',
#                                        'service_affinity', 'customer_service_segment',
#                                        'ps_tag_auto', 'ps_tag_link'
                                      ]]
df_customer_hyd.head(2)

,customer_id,apps_list,ltr,ltr_segment
8,62623de4056e7a73d6164f18,"[Amazon Prime Video, Wynk Music, BookMyShow, F...",147.0,PHH
9,65c3450bcd8e0a33c85f0322,"[Facebook, Messenger, Google Maps, Gmail, YouT...",1.0,HH


### Customer level apps list and category mapping 

In [20]:
## Explode list and map all the categories

df_customer_hyd_explode = df_customer_hyd.explode('apps_list')
df_customer_app_cat_mapping_v0 = pd.merge(df_customer_hyd_explode, category_list, 
                                       how='left', 
                                       left_on='apps_list', right_on='app_name')

df_customer_mapped_v0 = df_customer_app_cat_mapping_v0\
                        .groupby(['customer_id', 'ltr', 'ltr_segment'])\
                        .agg(apps_list = ('apps_list', set),
                             app_count = ('apps_list', 'nunique'),
                             categories_l1 = ('categories_l1', set),
                             categories_l1_count = ('categories_l1', 'nunique'),
                             categories_l2 = ('categories_l2', set),
                             categories_l2_count = ('categories_l2', 'nunique')
                            )\
                        .reset_index()

df_customer_mapped_v0['apps_list'] = df_customer_mapped_v0['apps_list'].apply(list)
df_customer_mapped_v0['categories_l1'] = df_customer_mapped_v0['categories_l1'].apply(list)
df_customer_mapped_v0['categories_l2'] = df_customer_mapped_v0['categories_l2'].apply(list)

print(df_customer_mapped_v0.shape)
df_customer_mapped_v0.head(5)

(145257, 9)


,customer_id,ltr,ltr_segment,apps_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count
0,573f28f59b0ffc283676fc3f,101.0,PHH,"[Swiggy, BookMyShow, Google Photos, Disney+ Ho...",29,"[Office, Educational, Streaming_Music, Wearabl...",13,"[Finance, Office, Educational, Wearable, Deliv...",9
1,573f28f59b0ffc283676fd0f,531.0,PHH,"[Swiggy, Google Photos, Disney+ Hotstar, Flipk...",28,"[Streaming_Music, Travel_Ridehailing, Finance_...",9,"[Finance, Delivery, Social, Entertainment, Eco...",7
2,573f28f59b0ffc283676feed,318.0,PHH,"[Swiggy, BookMyShow, Google Photos, Dropbox, L...",36,"[Dating, Office, Streaming_Music, Travel_Rideh...",15,"[Finance, Office, Wearable, Delivery, Social, ...",9
3,573f28f69b0ffc28367702b7,78.0,PHH,"[Swiggy, MS Authenticator, BookMyShow, Google ...",29,"[Office, Educational, Travel_Ridehailing, Fina...",11,"[Finance, Office, Educational, Delivery, Socia...",9
4,573f28f79b0ffc2836770aad,205.0,PHH,"[Swiggy, MS Authenticator, BookMyShow, Google ...",62,"[Wearable, Fantasy_Sports, Delivery_Grocery, D...",19,"[Finance, Office, Wearable, Delivery, Health, ...",12


### Sanity checks 

In [21]:
df_customer_mapped_v0.describe()

,ltr,app_count,categories_l1_count,categories_l2_count
count,145257.000000,145257.000000,145257.000000,145257.000000
mean,48.972724,21.450739,8.082454,6.798736
std,103.656239,8.997270,3.122646,2.135160
min,0.000000,1.000000,1.000000,1.000000
25%,3.000000,15.000000,6.000000,5.000000
50%,13.000000,20.000000,8.000000,7.000000
75%,48.000000,27.000000,10.000000,8.000000
max,5727.000000,85.000000,20.000000,14.000000


In [22]:
df_customer_mapped_v0[df_customer_mapped_v0['categories_l1_count'] == 1].head(5)

,customer_id,ltr,ltr_segment,apps_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count
11666,5d346fd2f8fc6575c088c0bc,24.0,PHH,"[Google Calendar, Google Photos, Adobe Scan, Y...",6,[Tools],1,[Tools],1
19641,5ddd80f16ffaa931412fa63a,84.0,PHH,"[Google Photos, Clock, YouTube, Gmail, Google ...",5,[Tools],1,[Tools],1
21012,5df9c189a692813d3c71b202,81.0,PHH,"[YouTube, Gmail, Google Maps]",3,[Tools],1,[Tools],1
30743,5fbcd08d90d7794ddfbe00f0,7.0,PHH,"[YouTube, Gmail, Google Maps]",3,[Tools],1,[Tools],1
32615,5ffcdca087d74172f6232e30,6.0,PHH,"[Google Calendar, Google Photos, YouTube, Gmai...",5,[Tools],1,[Tools],1


#### Distribution check on apps

In [32]:
total_customers = df_customer_app_cat_mapping_v0.customer_id.nunique()
total_customers


df_temp = df_customer_hyd_explode\
            .groupby(['apps_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,apps_list,customers,%
0,YouTube,147380,99.95
1,Google Maps,147351,99.93
2,Gmail,147301,99.89
3,WhatsApp,142551,96.67
4,Google Photos,139610,94.68
5,Instagram,113971,77.29
6,Facebook,98589,66.86
7,Truecaller,95221,64.58
8,Paytm Money,91590,62.11
9,Google Calendar,85845,58.22


#### Distribution check on categories

In [33]:
total_customers = df_customer_app_cat_mapping_v0.customer_id.nunique()
total_customers

df_temp = df_customer_app_cat_mapping_v0\
            .groupby(['categories_l1'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,customers,%
0,Tools,147447,99.99
1,Social,147122,99.77
2,OTT,110900,75.21
3,Ecommerce,105185,71.33
4,Finance_Transactions,98168,66.57
5,Streaming_Music,87892,59.61
6,Travel_Ridehailing,84556,57.34
7,Delivery_Food,81733,55.43
8,Office,72395,49.10
9,News,47972,32.53


#### Consider removing [Tools, Social] apps since it's almost present in all the customers mobile    

In [25]:
## Explode list and map all the categories and exclude Tools and Social apps

### Step 1
df_customer_app_cat_mapping_v1 = df_customer_app_cat_mapping_v0[~df_customer_app_cat_mapping_v0['categories_l1'].isin(['Tools','Social'])] 

### Step 2
df_customer_mapped_v1 = df_customer_app_cat_mapping_v1\
                        .groupby(['customer_id', 'ltr', 'ltr_segment'])\
                        .agg(apps_list = ('apps_list', set),
                             app_count = ('apps_list', 'nunique'),
                             categories_l1 = ('categories_l1', set),
                             categories_l1_count = ('categories_l1', 'nunique'),
                             categories_l2 = ('categories_l2', set),
                             categories_l2_count = ('categories_l2', 'nunique')
                            )\
                        .reset_index()
### Step 3
df_customer_mapped_v1['apps_list'] = df_customer_mapped_v1['apps_list'].apply(list)
df_customer_mapped_v1['categories_l1'] = df_customer_mapped_v1['categories_l1'].apply(list)
df_customer_mapped_v1['categories_l2'] = df_customer_mapped_v1['categories_l2'].apply(list)
print(df_customer_mapped_v1.shape)
df_customer_mapped_v1.head(5)

(142382, 9)


,customer_id,ltr,ltr_segment,apps_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count
0,573f28f59b0ffc283676fc3f,101.0,PHH,"[Swiggy, BookMyShow, Disney+ Hotstar, Flipkart...",20,"[Office, Educational, Streaming_Music, Wearabl...",11,"[Finance, Office, Educational, Wearable, Deliv...",7
1,573f28f59b0ffc283676fd0f,531.0,PHH,"[CRED, Ajio, Swiggy, Zomato, Ola, RedBus, Axis...",15,"[Streaming_Music, Travel_Ridehailing, Finance_...",7,"[Finance, Delivery, Entertainment, Ecommerce, ...",5
2,573f28f59b0ffc283676feed,318.0,PHH,"[Swiggy, BookMyShow, Dropbox, Flipkart, Cred, ...",25,"[Dating, Office, Streaming_Music, Travel_Rideh...",13,"[Finance, Office, Wearable, Delivery, Entertai...",7
3,573f28f69b0ffc28367702b7,78.0,PHH,"[Amazon Prime Video, Swiggy, Udemy, MS Authent...",17,"[Office, Educational, Travel_Ridehailing, Fina...",9,"[Finance, Office, Educational, Delivery, Enter...",7
4,573f28f79b0ffc2836770aad,205.0,PHH,"[Swiggy, MS Authenticator, BookMyShow, Slack, ...",45,"[Dating, Office, Streaming_Music, Travel_Rideh...",17,"[Finance, Office, Wearable, Delivery, Health, ...",10


##### Sanity checks 

In [26]:
df_customer_mapped_v0[df_customer_mapped_v0['customer_id'] == '573f28f59b0ffc283676fd0f']

,customer_id,ltr,ltr_segment,apps_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count
1,573f28f59b0ffc283676fd0f,531.0,PHH,"[Swiggy, Google Photos, Disney+ Hotstar, Flipk...",28,"[Streaming_Music, Travel_Ridehailing, Finance_...",9,"[Finance, Delivery, Social, Entertainment, Eco...",7


In [27]:
df_customer_mapped_v1[df_customer_mapped_v1['customer_id'] == '573f28f59b0ffc283676fd0f']

,customer_id,ltr,ltr_segment,apps_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count
1,573f28f59b0ffc283676fd0f,531.0,PHH,"[CRED, Ajio, Swiggy, Zomato, Ola, RedBus, Axis...",15,"[Streaming_Music, Travel_Ridehailing, Finance_...",7,"[Finance, Delivery, Entertainment, Ecommerce, ...",5


### HiPlot

In [28]:
df_customer_mapped_v0.columns

Index(['customer_id', 'ltr', 'ltr_segment', 'apps_list', 'app_count',
       'categories_l1', 'categories_l1_count', 'categories_l2',
       'categories_l2_count'],
      dtype='object')

### All categories

In [29]:
print('commented out code')

# hip.Experiment\
# .from_dataframe(df_customer_mapped_v0[['customer_id', # 'ltr', 
#                                        'ltr_segment', 'apps_list', 'app_count',
#                                        'categories_l1', 'categories_l1_count', 'categories_l2',
#                                        'categories_l2_count']])\
# .display(display_data_table=False)

commented out code


### Without Tools and Social

In [30]:
"""
hip.Experiment\
.from_dataframe(df_customer_mapped_v1[['customer_id', # 'ltr', 
                                       'ltr_segment', 'apps_list', 'app_count',
#                                        'categories_l1', 
#                                        'categories_l1_count', 
                                       'categories_l2',
                                       'categories_l2_count']])\
.display(display_data_table=False)
"""

"\nhip.Experiment.from_dataframe(df_customer_mapped_v1[['customer_id', # 'ltr', \n                                       'ltr_segment', 'apps_list', 'app_count',\n#                                        'categories_l1', \n#                                        'categories_l1_count', \n                                       'categories_l2',\n                                       'categories_l2_count']]).display(display_data_table=False)\n"